## Analytics

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw_2026 = pd.read_csv('/home/azureuser/cloudfiles/code/Users/813076/git/dark-wifi/nps_wifi_score_all2026.csv')

In [3]:
df_raw_2026.shape

(2050207, 20)

In [4]:
df_raw_2026.WIFI_SATISFACTION_SCR.mean()

np.float64(66.04274915269615)

In [6]:
df = df_raw_2026.copy()
if "SEG_DEP_DT" in df.columns:
    df["SEG_DEP_DT"] = pd.to_datetime(df["SEG_DEP_DT"], errors="coerce")
    df["DEP_MONTH"] = df["SEG_DEP_DT"].dt.to_period("M").astype(str)

SCORE_COLS = [
    "WIFI_SATISFACTION_SCR",
    "WIFI_SPEED_SCR",
    "WIFI_PRICE_SCR",
    "WIFI_RLBLTY_SCR",
    "WIFI_CNCT_EFFRT_SCR",
]
numeric_candidates = SCORE_COLS + ["LIKELIHOOD_RECOMMEND_SCL"]

def to_numeric_if_exists(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df



df = to_numeric_if_exists(df, numeric_candidates)
CATEGORY_COLS = [
    "NET_PROMO_CATG",
    "FLEET_CD",
    "SUBFLEET_CD",
    "CABIN_FLOWN_CD",
    "SEG_DEP_AIRPRT_IATA_CD",
    "SEG_ARVL_AIRPRT_IATA_CD",
]

def detect_wifi_scale(df: pd.DataFrame) -> tuple[str, float, float]:
    if "WIFI_SATISFACTION_SCR" not in df.columns or df["WIFI_SATISFACTION_SCR"].dropna().empty:
        return "unknown", np.nan, np.nan

    p99 = df["WIFI_SATISFACTION_SCR"].quantile(0.99)
    if p99 <= 5.5:
        return "1-5", 2, 4
    return "0-100", 40, 80



for c in CATEGORY_COLS:
    if c in df.columns:
        df[c] = df[c].astype("string")

scale_label, low_thr, high_thr = detect_wifi_scale(df)

In [9]:
df = df[
    (df["SEG_DEP_DT"] >= pd.Timestamp("2026-01-01"))
    & (df["SEG_DEP_DT"] <= pd.Timestamp("2026-08-21"))
]

In [10]:
df.shape

(2050207, 21)

In [7]:
def wifi_satisfaction_label(series: pd.Series) -> pd.Series:
    if series.dropna().empty:
        return pd.Series(index=series.index, dtype="object")

    max_val = series.quantile(0.99)
    if max_val <= 5.5:
        label_map = {
            1: "Extremely poor",
            2: "Poor",
            3: "Average",
            4: "Good",
            5: "Excellent",
        }
        return series.round().map(label_map)

    anchors = [1.0, 25.75, 50.5, 75.25, 100.0]
    labels = [
        "Extremely difficult",
        "Somewhat difficult",
        "Neutral",
        "Somewhat easy",
        "Extremely easy",
    ]

    def nearest_label(v):
        if pd.isna(v):
            return np.nan
        idx = int(np.argmin([abs(v - a) for a in anchors]))
        return labels[idx]

    return series.apply(nearest_label)

In [8]:
df_raw_2026['FLEET_CD'].value_counts()

FLEET_CD
737    626422
321    510600
EMJ    275914
CRJ    187887
319    171114
787     75252
320     74146
777     72676
ERJ     52792
BUS      3404
Name: count, dtype: int64